In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    PretrainedConfig, 
    PreTrainedTokenizerFast, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import f1_score, precision_score, recall_score

import zipfile
import json
import torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


In [ ]:
# ==========================================
# 1. PHASE 3 CONFIGURATION & LABELS
# ==========================================
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "A-240242/legal-gqa-checkpoints" # Assuming this is your Phase 1 GQA repo
OUTPUT_DIR = "./phase3_gqa_prefix"
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"
# Official Phase 3 Labels
LABEL_LIST = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", 
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", 
    "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"
]
NUM_LABELS = len(LABEL_LIST)
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}


In [ ]:
# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
def load_and_flatten_rr_data(split_dir, files):
    """Flattens list-based document JSONs into sentence-level (text, label) pairs."""
    sentences = []
    labels = []
    
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/{split_dir}/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # Iterate over the list of document dicts directly
        for doc in data:
            doc_texts = doc["text"]
            doc_labels = doc["labels"]
            for text, label in zip(doc_texts, doc_labels):
                sentences.append(text)
                labels.append(LABEL2ID[label])
                
    return sentences, labels

print("Downloading and processing Phase 3 Data...")
train_texts, train_labels = load_and_flatten_rr_data("Train", ["CL_train.json", "IT_train.json"])
dev_texts, dev_labels = load_and_flatten_rr_data("Train", ["CL_dev.json", "IT_dev.json"])
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"])
# CRITICAL: For Causal sequence classification, right padding is standard, 
# but we must extract the hidden state of the *actual last token* before padding.
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_data(texts, labels):
    tokenized = tokenizer(texts, padding=False, truncation=True, max_length=512)
    tokenized["labels"] = labels
    return Dataset.from_dict(tokenized)

train_dataset = tokenize_data(train_texts, train_labels)
eval_dataset = tokenize_data(dev_texts, dev_labels)

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

In [7]:
# ==========================================
# 3. LORMA GQA ARCHITECTURE FOR SEQUENCE CLASSIFICATION
# ==========================================
class GQAConfig(PretrainedConfig):
    model_type = "gqa"
    def __init__(self, vocab_size=32000, hidden_size=768, intermediate_size=2048, 
                 num_hidden_layers=12, num_attention_heads=12, num_key_value_heads=4, 
                 rms_norm_eps=1e-5, max_position_embeddings=2048, rope_theta=10000.0, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.rms_norm_eps = rms_norm_eps
        self.max_position_embeddings = max_position_embeddings
        self.rope_theta = rope_theta

class LormaAdapter(nn.Module):
    def __init__(self, hidden_size, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(hidden_size, rank, bias=False)
        self.B = nn.Linear(rank, hidden_size, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.scaling * self.B(self.A(x))
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000.0):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        t = torch.arange(max_position_embeddings, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x, seq_len=None):
        if seq_len > self.max_position_embeddings:
            self.max_position_embeddings = seq_len
            t = torch.arange(self.max_position_embeddings, device=x.device, dtype=torch.float32)
            freqs = torch.outer(t, self.inv_freq.to(x.device))
            emb = torch.cat((freqs, freqs), dim=-1)
            self.cos_cached = emb.cos().to(x.dtype)
            self.sin_cached = emb.sin().to(x.dtype)
            
        return (
            self.cos_cached[:seq_len, :].to(x.device),
            self.sin_cached[:seq_len, :].to(x.device)
        )

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(1)
    sin = sin.unsqueeze(0).unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class SwiGLUMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class GQAAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        self.head_dim = self.hidden_size // self.num_heads
        
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
        
        self.rotary_emb = RotaryEmbedding(self.head_dim, config.max_position_embeddings, config.rope_theta)

    def forward(self, hidden_states, attention_mask=None):
        bsz, q_len, _ = hidden_states.size()
        
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        cos, sin = self.rotary_emb(v, seq_len=q_len)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # Repeat KV heads to match query heads for GQA
        k = k.repeat_interleave(self.num_key_value_groups, dim=1)
        v = v.repeat_interleave(self.num_key_value_groups, dim=1)
        
        # Causal mask remains True since Phase 1 GQA was a causal model
        attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size)
        return self.o_proj(attn_output)
# --- Assume RMSNorm, RotaryEmbedding, SwiGLUMLP are imported/defined identical to Phase 2 ---
# (Omitted here for brevity, paste them from your Phase 2 GQA implementation)

# ==========================================
# 1. GQA ATTENTION WITH PREFIX INJECTION
# ==========================================
class GQAPrefixAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        self.head_dim = self.hidden_size // self.num_heads
        
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
        
        self.rotary_emb = RotaryEmbedding(self.head_dim, config.max_position_embeddings, config.rope_theta)

    def forward(self, hidden_states, prefix_k=None, prefix_v=None, attention_mask=None):
        bsz, q_len, _ = hidden_states.size()
        
        # 1. Project and transpose immediately to (bsz, num_heads, q_len, head_dim)
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # 2. Apply RoPE (RoPE expects sequence length at dim 2)
        cos, sin = self.rotary_emb(v, seq_len=q_len)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # 3. Inject Prefix Virtual Tokens AFTER RoPE
        if prefix_k is not None and prefix_v is not None:
            prefix_len = prefix_k.size(0)
            # Expand prefix: (prefix_len, num_kv_heads, head_dim) -> (bsz, num_kv_heads, prefix_len, head_dim)
            pk = prefix_k.unsqueeze(0).transpose(1, 2).expand(bsz, -1, -1, -1)
            pv = prefix_v.unsqueeze(0).transpose(1, 2).expand(bsz, -1, -1, -1)
            
            # Concat along the sequence length dimension (dim=2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
        else:
            prefix_len = 0

        # 4. Repeat KV heads for Grouped Query Attention (repeat along head dimension dim=1)
        k = k.repeat_interleave(self.num_key_value_groups, dim=1)
        v = v.repeat_interleave(self.num_key_value_groups, dim=1)
        
        # 5. Build Custom Causal + Prefix Mask
        total_k_len = q_len + prefix_len
        mask = torch.ones((q_len, total_k_len), dtype=torch.bool, device=q.device)
        mask = torch.tril(mask, diagonal=prefix_len)
        attn_mask = mask.view(1, 1, q_len, total_k_len)
        
        # 6. Compute Attention
        attn_output = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        
        # 7. Final Reshape
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size)
        return self.o_proj(attn_output)


class GQAPrefixDecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = GQAPrefixAttention(config)
        self.mlp = SwiGLUMLP(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, prefix_k=None, prefix_v=None, attention_mask=None):
        attn_out = self.self_attn(self.input_layernorm(hidden_states), prefix_k, prefix_v, attention_mask)
        hidden_states = hidden_states + attn_out
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states

# ==========================================
# 2. SEQUENCE CLASSIFIER WITH PREFIX TUNING
# ==========================================
class GQAPrefixForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels, num_virtual_tokens=16):
        super().__init__()
        self.config = config
        self.num_labels = num_labels
        self.num_virtual_tokens = num_virtual_tokens
        
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.layers = nn.ModuleList([GQAPrefixDecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.score = nn.Linear(config.hidden_size, num_labels, bias=False)
        
        # PREFIX TUNING PARAMETERS
        # Shape: (num_layers, 2 [for K and V], num_virtual_tokens, num_kv_heads, head_dim)
        head_dim = config.hidden_size // config.num_attention_heads
        self.prefix_params = nn.Parameter(
            torch.randn(config.num_hidden_layers, 2, num_virtual_tokens, config.num_key_value_heads, head_dim) * 1e-3
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        batch_size, sequence_length = input_ids.shape
        hidden_states = self.embed_tokens(input_ids)
        
        for i, layer in enumerate(self.layers):
            # Extract layer-specific prefix K and V
            prefix_k = self.prefix_params[i, 0]
            prefix_v = self.prefix_params[i, 1]
            hidden_states = layer(hidden_states, prefix_k, prefix_v, attention_mask)
            
        hidden_states = self.norm(hidden_states)
        logits = self.score(hidden_states)
        
        # Extract last non-padding token
        if self.config.pad_token_id is None:
            sequence_lengths = -1
        else:
            if input_ids is not None:
                sequence_lengths = torch.eq(input_ids, self.config.pad_token_id).int().argmax(-1) - 1
                sequence_lengths = sequence_lengths % input_ids.shape[-1]
                sequence_lengths = sequence_lengths.to(logits.device)
            else:
                sequence_lengths = -1

        pooled_logits = logits[torch.arange(batch_size, device=logits.device), sequence_lengths]

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(pooled_logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=pooled_logits)

In [8]:
# ==========================================
# 4. INITIALIZATION & WEIGHT TRANSFER
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing GQA LORMA Model for Rhetorical Role Prediction...")
config = GQAConfig(
    vocab_size=len(tokenizer), 
    pad_token_id=tokenizer.pad_token_id, 
    eos_token_id=tokenizer.eos_token_id
)

# Using 32 virtual tokens to give the model ample continuous context
prefix_model = GQAPrefixForSequenceClassification(config, num_labels=NUM_LABELS, num_virtual_tokens=32)

print("Fetching Phase 1 Pretrained GQA Checkpoint...")
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="model.safetensors", token=HF_TOKEN)
prefix_model.load_state_dict(load_file(model_weight_path), strict=False)

# Freeze entire backbone
for param in prefix_model.parameters(): 
    param.requires_grad = False

# Unfreeze ONLY the prefix parameters and the classification head
prefix_model.prefix_params.requires_grad = True
for param in prefix_model.score.parameters(): 
    param.requires_grad = True

prefix_model.to(device)

trainable_params = sum(p.numel() for p in prefix_model.parameters() if p.requires_grad)
print(f"Trainable Prefix Parameters: {trainable_params:,}")

Initializing GQA LORMA Model for Rhetorical Role Prediction...
Fetching Phase 1 Pretrained GQA Checkpoint...
Trainable Prefix Parameters: 206,592


In [9]:
# ==========================================
# 5. METRICS & TRAINING EXECUTION
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Primary metric for Phase 3 is Macro-F1[cite: 11]
    macro_f1 = f1_score(labels, predictions, average="macro")
    macro_p = precision_score(labels, predictions, average="macro", zero_division=0)
    macro_r = recall_score(labels, predictions, average="macro", zero_division=0)
    return {"macro_f1": macro_f1, "macro_precision": macro_p, "macro_recall": macro_r}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args_prefix = TrainingArguments(
    output_dir="./phase3_gqa_prefix",
    report_to="none",
    num_train_epochs=10, 
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4, 
    learning_rate=1e-2,  # Prefix tuning requires a highly aggressive learning rate
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    seed=42
)

prefix_trainer = Trainer(
    model=prefix_model,
    args=training_args_prefix,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("\nCommencing GQA Prefix Tuning for Rhetorical Role Prediction...")
prefix_trainer.train()


Commencing GQA Prefix Tuning for Rhetorical Role Prediction...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro F1,Macro Precision,Macro Recall
1,16.813693,4.609507,0.079867,0.242796,0.117632
2,15.096620,3.964254,0.187036,0.357536,0.198470
3,14.928042,4.060214,0.160211,0.262027,0.177807
4,14.345690,4.468350,0.148644,0.227072,0.175510
5,13.767736,4.639987,0.132294,0.242495,0.146974
6,13.533280,4.239896,0.164222,0.247340,0.177147
7,12.987813,3.973903,0.212277,0.315344,0.210972
8,12.554608,3.944002,0.199024,0.278801,0.202252
9,12.218318,3.875926,0.228603,0.291708,0.215411
10,11.752654,3.916529,0.222090,0.307496,0.210695


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=2640, training_loss=14.057919068769975, metrics={'train_runtime': 2257.9525, 'train_samples_per_second': 74.603, 'train_steps_per_second': 1.169, 'total_flos': 0.0, 'train_loss': 14.057919068769975, 'epoch': 10.0})

In [12]:
# ==========================================
# 6. VALIDATION SET EVALUATION
# ==========================================
print("Running evaluation on the Development Set...")
eval_results = prefix_trainer.evaluate()
print(f"Validation Macro-F1: {eval_results['eval_macro_f1']:.4f}")
print(f"Validation Precision: {eval_results['eval_macro_precision']:.4f}")
print(f"Validation Recall: {eval_results['eval_macro_recall']:.4f}")

Running evaluation on the Development Set...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Macro F1,Macro Precision,Macro Recall
11.752654,3.875926,10,0.228603,0.291708,0.215411


Validation Macro-F1: 0.2286
Validation Precision: 0.2917
Validation Recall: 0.2154


In [13]:
# ==========================================
# 7. LOAD TEST DATA
# ==========================================
def load_test_data(files):
    test_docs = []
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/Test/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            if isinstance(data, dict):
                for doc_id, content in data.items():
                    # If the content is just a list of sentences
                    if isinstance(content, list):
                        test_docs.append({"id": doc_id, "text": content})
                    # If it happens to be a dictionary
                    elif isinstance(content, dict):
                        if "id" not in content:
                            content["id"] = doc_id
                        test_docs.append(content)
                        
            elif isinstance(data, list):
                test_docs.extend(data)
                
    return test_docs

print("\nDownloading Test Data...")
test_docs = load_test_data(["CL_test.json", "IT_test.json"])
print(f"Successfully loaded {len(test_docs)} test documents.")

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

Successfully loaded 10 test documents.


In [14]:
# ==========================================
# 8. TEST SET INFERENCE & SUBMISSION
# ==========================================
prefix_model.eval()
prefix_model.to(device)
predictions = {}

print("Generating Rhetorical Role Predictions...")
for doc in tqdm(test_docs, desc="Predicting Document Sentences"):
    doc_id = doc["id"]
    sentences = doc["text"]
    doc_predictions = []
    
    batch_size = 16
    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i : i + batch_size]
        
        inputs = tokenizer(
            batch_sentences, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        ).to(device)
        
        with torch.inference_mode():
            outputs = prefix_model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            
        doc_predictions.extend([ID2LABEL[pred] for pred in preds])
        
    assert len(doc_predictions) == len(sentences), f"Count mismatch for {doc_id}"
    predictions[doc_id] = doc_predictions

Generating Rhetorical Role Predictions...


Predicting Document Sentences:   0%|          | 0/10 [00:00<?, ?it/s]

In [15]:
# ==========================================
# 9. SAVE & ZIP FOR CODABENCH
# ==========================================
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nPhase 3 Predictions complete! Submission saved to: {SUBMISSION_ZIP}")


Phase 3 Predictions complete! Submission saved to: ./phase3_gqa_prefix/submission.zip
